In [1]:
!pip install langchain chromadb faiss-cpu langchain-community wikipedia \
    langchain-huggingface sentence-transformers


  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
   ---------------------------------------- 0.0/16.5 MB ? eta -:--:--
   --- ------------------------------------ 1.6/16.5 MB 9.3 MB/s eta 0:00:02
   --------- ------------------------------ 3.9/16.5 MB 10.4 MB/s eta 0:00:02
   --------------- ------------------------ 6.3/16.5 MB 10.8 MB/s eta 0:00:01
   -------------------- ------------------- 8.7/16.5 MB 10.9 MB/s eta 0:00:01
   -------------------------- ------------- 11.0/16.5 MB 11.0 MB/s eta 0:00:01
   -------------------------------- ------- 13.4/16.5 MB 11.1 MB/s eta 0:00:01
   -------------------------------------- - 16.0/16.5 MB 11.2 MB/s eta 0:00:01
   ---------------------------------------- 16.5/16.


[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## WikipediaRetriever : 

In [3]:
from langchain_community.retrievers import WikipediaRetriever

In [4]:
# initialize the retriever ( optional : set language and top_k ) 

retriever = WikipediaRetriever(top_k_results = 2 , lang = "en")

In [5]:
# Define Your query : 

query = " the geopolitical history of India and pakistan from the perspective of a chinese"

# Get relevant Wikipedia document 
docs = retriever.invoke(query)

In [7]:
# Print retrieved content : 

for i , doc in enumerate(docs):
    print(f"\n---- Result {i+1} ----")
    print(f"Content : \n {doc.page_content}....")  # truncate for display


---- Result 1 ----
Content : 
 The India–Pakistan war of 1971, also known as the third Indo-Pakistani war, was a military confrontation between India and Pakistan that occurred during the Bangladesh Liberation War in East Pakistan from 3 December 1971 until the Pakistani capitulation in Dhaka on 16 December 1971.  The war began with Pakistan's Operation Chengiz Khan, consisting of preemptive aerial strikes on eight Indian air stations. The strikes led to India declaring war on Pakistan, marking their entry into the war for East Pakistan's independence, on the side of Bengali nationalist forces. India's entry expanded the existing conflict with Indian and Pakistani forces engaging on both the eastern and western fronts.
Thirteen days after the war started, India achieved a clear upper hand, and the Eastern Command of the Pakistan military signed the instrument of surrender on 16 December 1971 in Dhaka, marking the formation of East Pakistan as the new nation of Bangladesh. Approximatel

## Vector Store Retriever : 

In [8]:
from langchain_community.vectorstores import Chroma
from langchain_openai import OpenAIEmbeddings
from langchain_core.documents import Document

In [9]:
# Step 1 :  Your source documents 

documents = [
    Document(page_content = "Langchain helps developers build LLM applications easily."),
    Document(page_content = "Chroma is a vector database optimized for LLM-based search."),
    Document(page_content = "OpenAI provides powerfull embedding models."),
    ]

In [10]:
# Step 2 :  Initialize embedding model 

from langchain_huggingface import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# Step 3 : Create Chroma vector store in memory 

Vectorstore = Chroma.from_documents(
    documents = documents,
    embedding = embedding_model ,
    collection_name = "my_collection"
)

C:\Users\ANUBHAV\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4282.96it/s]


In [16]:
# Step 4 : Convert vectorstore into a retriever

retriever = Vectorstore.as_retriever(search_lkwargs = { "k" : 1})


In [17]:
query = "What is Chroma uesd for?"
results = retriever.invoke(query)



In [18]:
for i , doc in enumerate(results):
    print(f"\n-- Result {i + 1} -- ")
    print(doc.page_content)


-- Result 1 -- 
Chroma is a vector database optimized for LLM-based search.

-- Result 2 -- 
Langchain helps developers build LLM applications easily.

-- Result 3 -- 
OpenAI provides powerfull embedding models.


## Maximal Marginal Relevance (MMR) : 

In [22]:
# Sample documents
docs = [
    Document(page_content="LangChain makes it easy to work with LLMs."),
    Document(page_content="LangChain is used to build LLM based applications."),
    Document(page_content="Chroma is used to store and search document embeddings."),
    Document(page_content="Embeddings are vector representations of text."),
    Document(page_content="MMR helps you get diverse results when doing similarity search."),
    Document(page_content="LangChain supports Chroma, FAISS, Pinecone, and more."),
]

In [23]:
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings

# Initialize Hugging Face local embeddings
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# Create the FAISS vector store from documents
vectorstore = FAISS.from_documents(
    documents=docs,
    embedding=embedding_model
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 12462.52it/s]


In [27]:
# Enable MMR in the retrievel : 

retriever = vectorstore.as_retriever(
    search_type = "mmr",    #--> This enables MMR
    search_kwargs = { "k" : 3 , "lambda_mult" : 0.5}  #--> k = top results , lambda_mult = relevance-diversity balance
)

In [28]:
query = "what is langchain?"
results = retriever.invoke(query)

In [29]:
for i , doc in enumerate(results):
    print(f" \n -- Result {i + 1} --")
    print(doc.page_content)

 
 -- Result 1 --
LangChain supports Chroma, FAISS, Pinecone, and more.
 
 -- Result 2 --
LangChain is used to build LLM based applications.
 
 -- Result 3 --
Embeddings are vector representations of text.


## Multi-Query Retriever : 

In [34]:
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_ollama import ChatOllama
from langchain_classic.retrievers import MultiQueryRetriever


In [32]:
# Relevant health & wellness documents
all_docs = [
    Document(page_content="Regular walking boosts heart health and can reduce symptoms of depression.", metadata={"source": "H1"}),
    Document(page_content="Consuming leafy greens and fruits helps detox the body and improve longevity.", metadata={"source": "H2"}),
    Document(page_content="Deep sleep is crucial for cellular repair and emotional regulation.", metadata={"source": "H3"}),
    Document(page_content="Mindfulness and controlled breathing lower cortisol and improve mental clarity.", metadata={"source": "H4"}),
    Document(page_content="Drinking sufficient water throughout the day helps maintain metabolism and energy.", metadata={"source": "H5"}),
    Document(page_content="The solar energy system in modern homes helps balance electricity demand.", metadata={"source": "I1"}),
    Document(page_content="Python balances readability with power, making it a popular system design language.", metadata={"source": "I2"}),
    Document(page_content="Photosynthesis enables plants to produce energy by converting sunlight.", metadata={"source": "I3"}),
    Document(page_content="The 2022 FIFA World Cup was held in Qatar and drew global energy and excitement.", metadata={"source": "I4"}),
    Document(page_content="Black holes bend spacetime and store immense gravitational energy.", metadata={"source": "I5"}),
]

In [33]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

# Initialize local Hugging Face embeddings
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# Create FAISS vector store
vectorstore = FAISS.from_documents(
    documents=all_docs,
    embedding=embedding_model
)


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 31115.91it/s]


In [43]:
# Create retrievers [ similarity retrievel ] : 
similarity_retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 5})


llm = ChatOllama(
    model="llama3.2:1b",
    temperature=0
)

# Create retrievers [ Multi-query retrievel ] : 
multiquery_retriever = MultiQueryRetriever.from_llm(
    retriever=vectorstore.as_retriever(
        search_kwargs={"k": 5}
    ),
    llm=llm
)

In [36]:
# Query
query = "How to improve energy levels and maintain balance?"

In [44]:
# Retrieve results
similarity_results = similarity_retriever.invoke(query)
multiquery_results= multiquery_retriever.invoke(query)

In [45]:
for i, doc in enumerate(similarity_results):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content)

print("*"*150)

for i, doc in enumerate(multiquery_results):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content)


--- Result 1 ---
Drinking sufficient water throughout the day helps maintain metabolism and energy.

--- Result 2 ---
The solar energy system in modern homes helps balance electricity demand.

--- Result 3 ---
Consuming leafy greens and fruits helps detox the body and improve longevity.

--- Result 4 ---
Mindfulness and controlled breathing lower cortisol and improve mental clarity.

--- Result 5 ---
Photosynthesis enables plants to produce energy by converting sunlight.
******************************************************************************************************************************************************

--- Result 1 ---
Black holes bend spacetime and store immense gravitational energy.

--- Result 2 ---
Python balances readability with power, making it a popular system design language.

--- Result 3 ---
Photosynthesis enables plants to produce energy by converting sunlight.

--- Result 4 ---
Consuming leafy greens and fruits helps detox the body and improve longevity.

## ContextualCompressionRetriever : 

In [46]:
from langchain_community.vectorstores import FAISS

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_ollama import ChatOllama

from langchain_classic.retrievers import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import LLMChainExtractor

from langchain_core.documents import Document

In [47]:
# Recreate the document objects from the previous data
docs = [
    Document(page_content=(
        """The Grand Canyon is one of the most visited natural wonders in the world.
        Photosynthesis is the process by which green plants convert sunlight into energy.
        Millions of tourists travel to see it every year. The rocks date back millions of years."""
    ), metadata={"source": "Doc1"}),

    Document(page_content=(
        """In medieval Europe, castles were built primarily for defense.
        The chlorophyll in plant cells captures sunlight during photosynthesis.
        Knights wore armor made of metal. Siege weapons were often used to breach castle walls."""
    ), metadata={"source": "Doc2"}),

    Document(page_content=(
        """Basketball was invented by Dr. James Naismith in the late 19th century.
        It was originally played with a soccer ball and peach baskets. NBA is now a global league."""
    ), metadata={"source": "Doc3"}),

    Document(page_content=(
        """The history of cinema began in the late 1800s. Silent films were the earliest form.
        Thomas Edison was among the pioneers. Photosynthesis does not occur in animal cells.
        Modern filmmaking involves complex CGI and sound design."""
    ), metadata={"source": "Doc4"})
]

In [48]:
# Create a FAISS vector store from the documents
from langchain_huggingface import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vectorstore = FAISS.from_documents(
    docs,
    embedding_model
)


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5562.67it/s]


In [49]:
base_retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

In [51]:
# Set up the compressor using an LLM
from langchain_ollama import ChatOllama

llm = ChatOllama(
    model="llama3.2:1b",
    temperature=0
)

compressor = LLMChainExtractor.from_llm(llm)

In [52]:
# Create the contextual compression retriever
compression_retriever = ContextualCompressionRetriever(
    base_retriever=base_retriever,
    base_compressor=compressor
)


In [53]:
# Query the retriever
query = "What is photosynthesis?"
compressed_results = compression_retriever.invoke(query)

In [57]:
for i, doc in enumerate(compressed_results):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content)


--- Result 1 ---
Basketball was invented by Dr. James Naismith in the late 19th century.
